# 16S sequences processing

Notebook summary: Samples pre-processing: fetching, denoising, filtering, taxonomy annotation, and diversity (incl. k-mers) measures computation.

Author, date: Fannie Kerff, January-August 2025

Environment: qiime2-amplicon-2024.10 (saved in `environment.yml`)

## Setup

In [ ]:
# imports
import os
import shutil
import pandas as pd

from functions_script import process_qza_file, compute_stats_for_percentage_columns

## Import samples sequences and metadata

### [**IF ACCESS TO ALL STUDY DATA**] Import sequences and metadata from the 187* samples

*these are all samples from all infants in the study (after sequence data processing (below), only 163 samples will be included in the data analysis).

In [ ]:
# paths
data_path = "/cluster/work/bokulich/fkerff/GrumpyBiome-analysis"

meta_data_path = f"{data_path}/data-sensitive/meta_data"
seq_data_path = f"{data_path}/data-sensitive/seq_data"
output_path = f"{data_path}/data-sensitive/processed_data"

- Import sample metadata

In [ ]:
# define metadata file
meta_data_file = f"{meta_data_path}/raw/md_all.tsv"

- Import sequences data

In [ ]:
# import sequencing data
!mkdir -p qiime2/out
!sbatch --ntasks=1 --cpus-per-task=48 --time=120:00:00 --job-name="import-seqs" --mem-per-cpu=2048 --output="qiime2/out/0-import-seqs.out" --open-mode=append --wrap="qiime2/0-import-seqs.sh {data_path}"

In [ ]:
# !qiime2/0-import-seqs {seq_data_path}

### [**IF ONLY ACCESS TO PUBLIC DATA**] Import metadata from the 130* samples

*these are the samples related to published sequence data on ERA (in total, infants generated 149 samples - see their metadata in ../data/meta_data/raw/md_all.tsv).

In [ ]:
# paths
meta_data_path = "../data/meta_data"
seq_data_path = "../data/seq_data"
output_path = "../data/processed_data"

- Import sample metadata

In [ ]:
# define metadata file
metadata = pd.read_excel(f"{meta_data_path}/raw/metadata.xlsx", 
                         index_col=0, sheet_name='metadata_per_sample')

metadata = metadata[['infant_id', 'timepoint', 'age_days', 'sex', 'sample_id']]
metadata['infant_id'] = 'i_' + metadata['infant_id'].astype(str)

metadata.to_csv(f"{meta_data_path}/md_proc.tsv", sep='\t', index=True, index_label="id")
meta_data_file = f"{meta_data_path}/md_proc.tsv"

- Import sequences data

In [ ]:
# fetch sequencing data with `q2-fondue` and ERA run accessions from PRJEB104111
!mkdir -p {seq_data_path}
!mkdir -p qiime2/out
!sbatch --ntasks=1 --cpus-per-task=48 --time=15:00:00 --job-name="fetch-seqs" --mem-per-cpu=2048 --output="qiime2/out/0-fetch-seqs.out" --open-mode=append --wrap="qiime2/0-fetch-seqs.sh {meta_data_path} {seq_data_path}"

In [ ]:
# !qiime2/0-fetch-seqs.sh {meta_data_path} {seq_data_path}

## Visualize fetched sequence data

In [ ]:
# visualize fetched sequences
!mkdir -p {output_path}/0-import-demux
!sbatch --ntasks=1 --cpus-per-task=12 --time=2:00:00 --job-name="import-demux" --mem-per-cpu=1024 --output="qiime2/out/0.2-import-demux.out" --open-mode=append --wrap="qiime2/0.2-import-demux.sh {seq_data_path} {output_path}"

In [ ]:
# !qiime2/0.2-import-demux.sh {seq_data_path} {output_path}

## Denoising sequence data with DADA2

In [ ]:
# denoise sequences with dada2
!mkdir -p {output_path}/1-denoise
!sbatch --ntasks=1 --cpus-per-task=12 --time=24:00:00 --job-name="denoise" --mem-per-cpu=2048 --output="qiime2/out/1-denoise.out" --open-mode=truncate --wrap="qiime2/1-denoise.sh {output_path} 150 150 {meta_data_file} {seq_data_path}"

In [ ]:
# !qiime2/1-denoise.sh {output_path} 150 150 {meta_data_file} {seq_data_path}

In [ ]:
# check dada2 stats
process_qza_file(f"{output_path}/1-denoise/dada2-stats.qza", compute_stats_for_percentage_columns)

## Filtering based on sampling depth

1. ### Defining sampling depth

In [ ]:
# define sampling depth using alpha rarefaction plot with a max-depth at the median (26,922 bp)
!mkdir -p {output_path}/1.2-rarefaction
!sbatch --ntasks=1 --cpus-per-task=12 --time=24:00:00 --job-name="rarefaction" --mem-per-cpu=2048 --output="qiime2/out/2-rarefaction.out" --open-mode=truncate --wrap="qiime2/2-alpha-rarefaction.sh {output_path} {meta_data_file} 26922 20"

In [ ]:
# !qiime2/2-alpha-rarefaction.sh {output_path} {meta_data_file} 26922 10

The shannon diversity along sequencing depths file* was saved into `{meta_data_path}/raw/shannon.csv`.

*Steps to download the `shannon.csv` output file: go to [QIIME view](https://view.qiime2.org/), drop the output file `{output_path}/1.2-rarefaction/alpha-rarefaction-plot-20.qzv`, make sure that `Metric: shannon` and `Sample Metadata Column: sample_id`, then click on `Download CSV`.

2. ### Filtering

In [ ]:
# exclude samples with a total frequency less than the sampling depth of 3035
!mkdir -p {output_path}/2-filter-3035
!sbatch --ntasks=1 --cpus-per-task=12 --time=2:00:00 --job-name="filter-samples" --mem-per-cpu=1024 --output="qiime2/out/3-filter-samples.out" --open-mode=truncate --wrap="qiime2/3-filter-samples.sh {output_path} 3035 {meta_data_file}"

In [ ]:
# !qiime2/3-filter-samples.sh {output_path} 3035 {meta_data_file}

In [ ]:
# exclude features with a total abundance < 10; exclude sequences not related to filtered features
!sbatch --ntasks=1 --cpus-per-task=12 --time=2:00:00 --job-name="filter-features" --mem-per-cpu=1024 --output="qiime2/out/3.2-filter-features.out" --open-mode=truncate --wrap="qiime2/3.2-filter-features.sh {output_path} 3035 {meta_data_file} 1 10"

In [ ]:
# !qiime2/3.2-filter-features.sh {output_path} 3035 {meta_data_file} 1 10

## Taxonomic annotation of ASVs/observed sequences

In [ ]:
# download SILVA 138.1 database files for classifier training (to be run locally)
"""qiime rescript get-silva-data \
    --p-version '138.1' \
    --p-target 'SSURef_NR99' \
    --o-silva-sequences "../silva-138.1-ssu-nr99-rna-seqs.qza" \
    --o-silva-taxonomy "../silva-138.1-ssu-nr99-tax.qza" \
    --verbose"""

In [ ]:
# the two output files were copied in the folder 
# /cluster/work/bokulich/fkerff/GrumpyBiome-analysis/seq_data/3-silva-classifier
silva_path = "/cluster/work/bokulich/fkerff/GrumpyBiome-analysis/seq_data/3-silva-classifier"

In [ ]:
# create silva classifier for the 16S V4 region
!mkdir -p {output_path}/3-silva-classifier
!sbatch --ntasks=1 --cpus-per-task=48 --time=24:00:00 --job-name="silva-classifier" --mem-per-cpu=1024 --output="qiime2/out/4-silva-classifier.out" --open-mode=truncate --wrap="qiime2/4-get_silva_reads_n_classifier.sh {silva_path}"

In [ ]:
# !qiime2/4-get_silva_reads_n-classifier.sh {silva_path}

In [ ]:
# define path to silva classifier
silva_classifier_path = f"{silva_path}/silva-138.1-ssu-nr99-515f-806r-classifier.qza"

In [ ]:
# perform taxonomic annotation of ASVs/observed sequences
!mkdir -p {output_path}/4-taxonomy-3035
!sbatch --ntasks=1 --cpus-per-task=12 --time=2:00:00 --job-name="feature-classifier" --mem-per-cpu=1024 --output="qiime2/out/5-feature-classifier.out" --open-mode=truncate --wrap="qiime2/5-feature-classifier.sh {output_path} 3035 {silva_classifier_path}"

In [ ]:
# !qiime2/5-feature-classifier.sh {output_path} 3035 {silva_path}

## Taxonomic filtering

In [ ]:
# remove ASVs that were not assigned a phylum, or host-related sequences
# (e.g., chloroplasts, mitochondria)
!sbatch --ntasks=1 --cpus-per-task=12 --time=2:00:00 --job-name="filter-taxa" --mem-per-cpu=1024 --output="qiime2/out/6-filter-taxa.out" --open-mode=truncate --wrap="qiime2/6-filter-taxa.sh {output_path} {meta_data_file} 3035"

In [ ]:
# !qiime2/6-filter-taxa.sh {output_path} {meta_data_file} 3035

## Microbial composition: barplots and phylogenetic tree

In [ ]:
# generate taxonomic composition barplots
!sbatch --ntasks=1 --cpus-per-task=12 --time=2:00:00 --job-name="taxa-barplot" --mem-per-cpu=1024 --output="qiime2/out/7-taxa-barplot.out" --open-mode=truncate --wrap="qiime2/7-taxa-barplot.sh {output_path} {meta_data_file} 3035"

In [ ]:
# !qiime2/7-taxa-barplot.sh {output_path} {meta_data_file} 3035

In [ ]:
# build a phylogenetic tree from our ASV sequences
directory = f"{output_path}/5-phylogeny-3035"

if os.path.isdir(directory):
    print(f"Directory {directory} exists. Deleting it...")
    shutil.rmtree(directory)
    print(f"Directory {directory} has been deleted.")

!sbatch --ntasks=1 --cpus-per-task=12 --time=2:00:00 --job-name="build-phylogenetic-tree" --mem-per-cpu=1024 --output="qiime2/out/8-build-phylogenetic-tree.out" --open-mode=truncate --wrap="qiime2/8-build-phylogenetic-tree.sh {output_path} 3035"

In [ ]:
# !qiime2/8-build-phylogenetic-tree.sh {output_path} 3035

In [ ]:
# visualize the phylogenetic tree with metadata and empress plugin
!sbatch --ntasks=1 --cpus-per-task=12 --time=2:00:00 --job-name="viz-phylogenetic-tree" --mem-per-cpu=1024 --output="qiime2/out/9-viz-phylogenetic-tree.out" --open-mode=truncate --wrap="qiime2/9-viz-phylogenetic-tree.sh {output_path} {meta_data_file} 3035"

In [ ]:
# !qiime/9-viz-phylogenetic-tree.sh {output_path} {meta_data_file} 3035

Then use the [EMPress plugin](https://old-library.qiime2.org/plugins/empress/32/) to visualise the tree using [QIIME view](https://view.qiime2.org/).

## Diversity metrics

In [ ]:
directory = f"{output_path}/6-diversity-3035"

if os.path.isdir(directory):
    print(f"Directory {directory} exists. Deleting it...")
    shutil.rmtree(directory)
    print(f"Directory {directory} has been deleted.")

# compute diversity metrics
!sbatch --ntasks=1 --cpus-per-task=12 --time=2:00:00 --job-name="diversities" --mem-per-cpu=1024 --output="qiime2/out/10-diversities.out" --open-mode=truncate --wrap="qiime2/10-diversities.sh {output_path} {meta_data_file} 3035 100 3033"

In [ ]:
# !qiime/10-diversities.sh {output_path} {meta_data_file} 3035 100 3033

## K-mers

### Frequency

In [ ]:
# perform k-mer based classification with k=10, 20 and 50
!mkdir -p {output_path}/7-kmer-results-3035
!sbatch --ntasks=1 --cpus-per-task=48 --time=24:00:00 --job-name="kmer" --mem-per-cpu=1024 --output="qiime2/out/11-kmer.out" --open-mode=truncate --wrap="qiime2/11-kmer.sh {output_path} 3035 16 10"
# edit the previous line to change k-mer size (last argument)

In [ ]:
# !qiime/11-kmer.sh {output_path} 3035 16 10
# edit the previous line to change k-mer size (last argument)

### Diversity

In [ ]:
base_dir = f"{output_path}/7-kmer-results-3035/diversity-kmer"
sub_dirs = ["alpha-diversities",
            "beta-div-distance-matrices",
            "pcoas-beta-div-metrics"]

for sub_dir in sub_dirs:
    full_path = os.path.join(base_dir, sub_dir)
    os.makedirs(full_path, exist_ok=True)
    print(f"Created or verified: {full_path}")

In [ ]:
# compute k-mer based diversity metrics
!sbatch --ntasks=1 --cpus-per-task=48 --time=2:00:00 --job-name="diversities-kmer" --mem-per-cpu=1024 --output="qiime2/out/11.2-diversities-kmer.out" --open-mode=truncate --wrap="qiime2/11.2-diversities-kmer.sh {output_path} {meta_data_file} 3035 16 3033"

In [ ]:
# !qiime/11.2-diversities-kmer.sh {output_path} {meta_data_file} 3035 16 3033

## Microbiome composition: phylum, family and genus-level agglomeration

In [ ]:
# perform taxonomy-level agglomeration: genus level
!mkdir -p {output_path}/9-comp-3035
!sbatch --ntasks=1 --cpus-per-task=48 --time=24:00:00 --job-name="comp-genus" --mem-per-cpu=1024 --output="qiime2/out/12-comp-genus.out" --open-mode=truncate --wrap="qiime2/12-composition.sh {output_path} 6 genus 3035 {meta_data_file}"

In [ ]:
# !qiime/12-composition.sh {output_path} 6 genus 3035 {meta_data_file}

In [ ]:
# perform taxonomy-level agglomeration: family level
!mkdir -p {output_path}/9-comp-3035
!sbatch --ntasks=1 --cpus-per-task=48 --time=24:00:00 --job-name="comp-family" --mem-per-cpu=1024 --output="qiime2/out/12-comp-family.out" --open-mode=truncate --wrap="qiime2/12-composition.sh {output_path} 5 family 3035 {meta_data_file}"

In [ ]:
# !qiime/12-composition.sh {output_path} 5 family 3035 {meta_data_file}

In [ ]:
# perform taxonomy-level agglomeration: phylum level
!sbatch --ntasks=1 --cpus-per-task=48 --time=24:00:00 --job-name="comp-phylum" --mem-per-cpu=1024 --output="qiime2/out/12-comp-phylum.out" --open-mode=truncate --wrap="qiime2/12-composition.sh {output_path} 2 phylum 3035 {meta_data_file}"

In [ ]:
# !qiime/12-composition.sh {output_path} 2 phylum 3035 {meta_data_file}

## Relative abundance tables

In [ ]:
# features level (for the random forest)
!mkdir -p {output_path}/8-rel-freq-3035
!sbatch --ntasks=1 --cpus-per-task=48 --time=24:00:00 --job-name="rel-abund" --mem-per-cpu=1024 --output="qiime2/out/13-rel-abund.out" --open-mode=truncate --wrap="qiime2/13-rel-abund.sh {output_path} 3035 2-filter taxa-filtered rel-freq"

In [ ]:
# !qiime/13-rel-abund.sh {output_path} 3035 2-filter taxa-filtered rel-freq

In [ ]:
# genus level (for individual genera rhythmicities)
!sbatch --ntasks=1 --cpus-per-task=48 --time=24:00:00 --job-name="rel-abund-genus" --mem-per-cpu=1024 --output="qiime2/out/13-rel-abund-genus.out" --open-mode=truncate --wrap="qiime2/13-rel-abund.sh {output_path} 3035 9-comp genus genus-rel-freq"

In [ ]:
# !qiime/13-rel-abund.sh {output_path} 3035 4-taxonomy genus genus-rel-freq

## Differential abundance analysis on melatonin

In [ ]:
# define samples' melatonin metadata file
metadata_mel = pd.read_excel(f"{meta_data_path}/raw/metadata.xlsx", 
                         index_col=0, sheet_name='metadata_per_sample')

metadata_mel.rename(columns={'Melatonin in Stool (pg/g)': 'melatonin'}, inplace=True)
metadata_mel = metadata_mel[['infant_id', 'melatonin', 'age_days', 'sex']]

metadata_mel.to_csv(f"{meta_data_path}/md_mel.tsv", sep='\t', index=True, index_label="id")
meta_data_mel_file = f"{meta_data_path}/md_mel.tsv"

In [ ]:
# genus level differential abundance analysis with ANCOM-BC2
# note that one needs to be on env. qiime2-2025.7 to include ancombc2
!sbatch --ntasks=1 --cpus-per-task=48 --time=24:00:00 --job-name="diff-abund-genus" --mem-per-cpu=1024 --output="qiime2/out/14-diff-abund-genus.out" --open-mode=truncate --wrap="qiime2/14-diff-abund.sh {output_path} {meta_data_mel_file} 3035 9-comp genus BH 0.1"

In [ ]:
# !qiime/14-diff-abund.sh {output_path} {meta_data_mel_file} 3035 9-comp genus BH 0.1

In [ ]:
# family level differential abundance analysis with ANCOM-BC2
# note that one needs to be on env. qiime2-2025.7 to include ancombc2
!sbatch --ntasks=1 --cpus-per-task=48 --time=24:00:00 --job-name="diff-abund-family" --mem-per-cpu=1024 --output="qiime2/out/14-diff-abund-family.out" --open-mode=truncate --wrap="qiime2/14-diff-abund.sh {output_path} {meta_data_mel_file} 3035 9-comp family BH 0.1"

In [ ]:
# !qiime/14-diff-abund.sh {output_path} {meta_data_mel_file} 3035 9-comp family BH 0.1